In [1]:
import pandas as pd
from geopy.distance import geodesic
from math import radians, sin, cos, asin, sqrt

CELL 1: ĐỌC DỮ LIỆU

In [2]:
print("Đang đọc dữ liệu...")
df_orders = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\orders.csv')
df_customers = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\customers.csv')
df_items = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\order_items.csv')
df_sellers = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\sellers.csv')
df_geo = pd.read_csv(r'D:\Delivery_Time_Prediction_project\data\raw\geolocation.csv')

print("Đã đọc xong dữ liệu")    
print(f"Orders: {df_orders.shape}")
print(f"Items: {df_items.shape}")
print(f"Customers:{df_customers.shape}")
print(f"Sellers: {df_sellers.shape}")
print(f"Geolocation: {df_geo.shape}")

Đang đọc dữ liệu...
Đã đọc xong dữ liệu
Orders: (99441, 8)
Items: (112650, 7)
Customers:(99441, 5)
Sellers: (3095, 4)
Geolocation: (1000163, 5)


CELL 2: KIỂM TRA DỮ LIỆU TRƯỚC KHI JOIN

In [3]:
is_orders_unique = df_orders['order_id'].is_unique
print("Cột 'order_id' có trùng lặp không?")
if is_orders_unique is True:
    print("Order_id không có giá trị trùng lặp.")
is_geo_unique = df_geo['geolocation_zip_code_prefix'].is_unique
print("Cột zip_code của Geo có độc nhất?", is_geo_unique)

if not is_geo_unique:
    print("Có dữ liệu trùng lặp trong bảng Geo")
    print("Top 5 mã bưu điện bị trùng lặp nhiều nhất", df_geo['geolocation_zip_code_prefix'].value_counts().head(5))

Cột 'order_id' có trùng lặp không?
Order_id không có giá trị trùng lặp.
Cột zip_code của Geo có độc nhất? False
Có dữ liệu trùng lặp trong bảng Geo
Top 5 mã bưu điện bị trùng lặp nhiều nhất geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
Name: count, dtype: int64


CELL 3: JOIN CÁC BẢNG VÀ XỬ LÝ MÃ BƯU ĐIỆN

In [4]:
print("Step 1: Tính aggregated features...")
order_summary = df_items.groupby(['order_id','seller_id']).agg({
    'order_item_id': 'count',        
    'price': 'sum',                  
    'freight_value': 'sum',          
    'seller_id': 'nunique'           
}).rename(columns={
    'order_item_id': 'num_items',
    'price': 'total_price',
    'freight_value': 'total_freight',
    'seller_id': 'num_sellers'
}).reset_index()

print(f"Order_summary: {order_summary.shape}")

print("step 2: Merging...")
# Join bảng orders với order_items
df_merged = pd.merge(df_orders, order_summary, on='order_id', how='inner')
print(f"Sau khi merge orders với order_summary: {df_merged.shape}")
# Join bảng vừa tạo với bảng khách hàng
df_merged = pd.merge(df_merged, df_customers, on='customer_id', how='inner')
print(f"Sau khi merge df_merged với df_customer: {df_merged.shape}")
# Join bảng vừa tạo với bảng seller
df_merged = pd.merge(df_merged, df_sellers, on='seller_id', how='inner')
print(f"Sau khi merge df_merged với df_sellers: {df_merged.shape}")
print("Step 3: Xử lý geolocation...")
#Gom nhóm các tọa độ trùng lặp có cùng 1 mã bưu điện và lấy mean
df_geo_grouped = df_geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].mean().reset_index()

df_merged = pd.merge(df_merged, df_geo_grouped, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df_merged.rename(columns={'geolocation_lat': 'customer_lat','geolocation_lng':'customer_lng'}, inplace=True)
df_merged.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

df_merged = pd.merge(df_merged, df_geo_grouped, left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix', how='left')
df_merged.rename(columns={'geolocation_lat':'seller_lat','geolocation_lng': 'seller_lng'}, inplace=True)
df_merged.drop('geolocation_zip_code_prefix',axis=1, inplace=True)

print("Số dòng sau khi merge hoàn tất: ", df_merged.shape)


Step 1: Tính aggregated features...
Order_summary: (100010, 6)
step 2: Merging...
Sau khi merge orders với order_summary: (100010, 13)
Sau khi merge df_merged với df_customer: (100010, 17)
Sau khi merge df_merged với df_sellers: (100010, 20)
Step 3: Xử lý geolocation...
Số dòng sau khi merge hoàn tất:  (100010, 24)


CELL 4: LÀM SẠCH DỮ LIỆU

In [5]:
#Lọc đơn hàng
df_clean = df_merged[df_merged['order_status']=='delivered'].copy()
#Xóa các dữ liệu khuyết
df_clean.dropna(subset=['order_delivered_customer_date'], inplace=True)
df_clean.dropna(subset=['customer_lat','customer_lng','seller_lat','seller_lng'], inplace=True)

#Chuyển đổi định dạng thời gian

date_columns = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col])

print("Đã dọn dẹp xong!")
print(f"Số đơn hàng hợp lệ:  {df_clean.shape[0]} dòng")

Đã dọn dẹp xong!
Số đơn hàng hợp lệ:  97325 dòng


CELL 5: TẠO THÊM BẢNG

In [6]:
#Tạo bảng delivery_time
df_clean['delivery_time'] = (df_clean['order_delivered_customer_date'] - df_clean['order_approved_at']).dt.total_seconds()/(24*3600)


print(f"Mean: {df_clean['delivery_time'].mean():.2f} days")
print(f"Min: {df_clean['delivery_time'].min():.2f} days")
print(f"Max: {df_clean['delivery_time'].max():.2f} days")
#Công thức haversine để tính khoảng cách 2 điểm 

def haversine_distance(lat1, lon1, lat2, lon2):

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2*asin(sqrt(a))
    r = 6371

    return c*r

df_clean['distance_km'] = df_clean.apply(lambda row: haversine_distance(row['seller_lat'], row['seller_lng'], row['customer_lat'], row['customer_lng']), axis = 1)
print("distance_km calculator")
print(f" Mean: {df_clean['distance_km'].mean():.2f}")
print(f" Max: {df_clean['distance_km'].max():.2f}")
print(f" Min: {df_clean['distance_km'].min():.2f}")




Mean: 12.07 days
Min: -6.99 days
Max: 208.50 days
distance_km calculator
 Mean: 600.26
 Max: 8677.91
 Min: 0.00


CELL 6: FINAL CHECK AND SAVE

In [7]:
print("✅ PREPROCESSING COMPLETED!")
print("="*60)
print(f"Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"\nColumns:")
print(df_clean.columns.tolist())
print(f"\nMissing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\nFirst 5 rows:")
print(df_clean.head())
 
# ============= CELL 12: Save =============
print("\n💾 Saving...")
df_clean.to_csv(r'D:\Delivery_Time_Prediction_project\data\processed\step2_cleaned_data.csv', index=False)
print("✅ Data saved to 'step2_cleaned_data.csv'")

✅ PREPROCESSING COMPLETED!
Final shape: 97325 rows × 26 columns

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'seller_id', 'num_items', 'total_price', 'total_freight', 'num_sellers', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'delivery_time', 'distance_km']

Missing values:
order_approved_at               14
order_delivered_carrier_date     1
delivery_time                   14
dtype: int64

First 5 rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d

In [8]:
#Check again
print(df_clean[['order_id', 'order_approved_at', 'order_delivered_customer_date', 'delivery_time']].head(20))


                            order_id   order_approved_at  \
0   e481f51cbdc54678b7cc49136f2d6af7 2017-10-02 11:07:15   
1   53cdb2fc8bc7dce0b6741e2150273451 2018-07-26 03:24:27   
2   47770eb9100c2d0c44946d9cf07ec65d 2018-08-08 08:55:23   
3   949d5b44dbf5de918fe9c16f97b45f8a 2017-11-18 19:45:59   
4   ad21c59c0840e6cb83a9ceb5573f8159 2018-02-13 22:20:29   
5   a4591c265e18cb1dcee52889e2d8acc3 2017-07-09 22:10:13   
7   6514b8ad8028c9f2cc2374ded245783f 2017-05-16 13:22:11   
8   76c6e866289321a7c93b82b54852dc33 2017-01-25 02:50:47   
9   e69bfb5eb88e0ed6a785585b27e16dbf 2017-07-29 12:05:32   
10  e6ce16cb79ec1d90b1da9085a6118aeb 2017-05-16 19:50:18   
11  34513ce0c4fab462a55830c0989c7edb 2017-07-13 20:10:08   
12  82566a660a982b15fb86e904c8d32918 2018-06-09 03:13:12   
13  5ff96c15d0b717ac6ad1f3d77225a350 2018-07-25 17:55:14   
14  432aaf21d85167c2c86ec9448c4e42cc 2018-03-01 15:10:47   
15  dcb36b511fcac050b97cd5c05de84dc3 2018-06-12 23:31:02   
16  403b97836b0c04a622354cf531062e5f 201

In [9]:
#KIỂM TRA LỖI 
neg_rows = df_clean[df_clean['delivery_time'] < 0]
print(f"Negative row {len(neg_rows)}")
print(neg_rows[['order_id', 'order_approved_at', 'order_delivered_customer_date', 'delivery_time']])


Negative row 63
                               order_id   order_approved_at  \
205    58d4c4747ee059eeeb865b349b41f53a 2018-07-26 23:31:53   
490    4df92d82d79c3b52c7138679fa9b07fc 2018-07-29 23:30:52   
2011   6e57e23ecac1ae881286657694444267 2018-08-20 15:55:42   
3692   f222c56f035b47dfa1e069a88235d730 2018-02-04 23:31:47   
11797  cf72398d0690f841271b695bbfda82d2 2017-09-13 22:04:39   
...                                 ...                 ...   
90207  fcbf4f4ef049367f9f85af94ed3b6010 2018-04-24 18:41:20   
92332  4387477eec4b3c89b39f3f454940d059 2018-08-20 15:56:29   
94220  4f3a6e28d764cf896b1fceb0028422c8 2018-07-05 16:21:50   
94905  9c3186381b733d4304e2e416afc6bbc1 2018-08-02 23:30:29   
98929  5a41aefdf8010bbd69a5264f69213b73 2018-07-05 16:17:20   

      order_delivered_customer_date  delivery_time  
205             2018-07-25 23:58:19      -0.981644  
490             2018-07-27 18:55:57      -2.190914  
2011            2018-08-17 16:45:45      -2.965243  
3692           

DỮ LIỆU BỊ SAI , CÓ VẤN ĐỀ KHI MÀ NGÀY KHÁCH NHẬN ĐƯỢC HÀNG LẠI TRƯỚC KHI SHOP DUYỆT ĐƠN

In [10]:
df_clean = df_clean[df_clean['delivery_time'] > 0]
print(f" Sau khi xóa các dữ liệu lỗi: {df_clean.shape}")

print("Saving...")
df_clean.to_csv(r'D:\Delivery_Time_Prediction_project\data\processed\step2_cleaned_data.csv', index=False)
print("Đã lưu lại file step 2")

 Sau khi xóa các dữ liệu lỗi: (97248, 26)
Saving...
Đã lưu lại file step 2
